# Bronze para Silver — geração distribuída

Uma linha por UC com solar. A Bronze entra como texto. Esta Silver converte potência, injeção e data, corrige a escala acima de 200 kWp e marca a conexão anterior à ligação da UC.

A média injetada fica na linha para classificar a UC. Ela não é subtraída do consumo nem da fatura. A Gold é quem monta `possui_gd` a partir do mês de `data_conexao`.

As regras estão em `docs/auditoria_dados/silver/silver_geracao_distribuida.md`.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DecimalType

catalog = "v-energy"
bronze_gd = f"`{catalog}`.bronze.tb_geracao_distribuida"
bronze_uc = f"`{catalog}`.bronze.tb_unidades_consumidoras"
silver_gd = f"`{catalog}`.silver.tb_geracao_distribuida"
quarentena_gd = f"`{catalog}`.silver.tb_geracao_distribuida_quarentena"

DATA_MIN = "2024-01-01"
DATA_MAX = "2026-06-30"
ESCALA_ACIMA_DE = 200
UCS_DATA_INCONSISTENTE = [
    "UC100259",
    "UC102507",
    "UC106046",
    "UC110560",
    "UC119180",
]

MODALIDADE = {
    "GERACAO NA PROPRIA UC": "PROPRIA_UC",
    "AUTOCONSUMO REMOTO": "AUTOCONSUMO_REMOTO",
    "GERACAO COMPARTILHADA": "COMPARTILHADA",
}

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.silver")
print(bronze_gd)
print(silver_gd)


`v-energy`.bronze.tb_geracao_distribuida
`v-energy`.silver.tb_geracao_distribuida


In [ ]:
mapa_modalidade = F.create_map(
    [item for par in MODALIDADE.items() for item in (F.lit(par[0]), F.lit(par[1]))]
)

# A Bronze de UC traz o cabeçalho repetido numa linha: data_ligacao vale o texto "data_ligacao".
# try_cast devolve nulo se a data não converter, em vez de interromper a carga.
df_uc = (
    spark.table(bronze_uc)
    .filter(F.trim(F.col("id_uc")) != F.lit("id_uc"))
    .select(
    F.trim(F.col("id_uc")).alias("uc_id"),
    F.try_to_date(F.trim(F.col("data_ligacao"))).alias("data_ligacao"),
    F.col("classe").alias("classe"),
    F.col("municipio").alias("municipio"),
    F.col("ingestion_datetime").alias("uc_ingestion_datetime"),
    )
)
janela_uc = Window.partitionBy("uc_id").orderBy(
    F.col("uc_ingestion_datetime").desc_nulls_last()
)
df_uc = (
    df_uc.withColumn("_rn", F.row_number().over(janela_uc))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "uc_ingestion_datetime")
)

df = (
    spark.table(bronze_gd)
    .select(
        F.trim(F.col("id_uc")).alias("id_uc"),
        F.trim(F.col("potencia_kwp")).alias("potencia_kwp_raw"),
        F.expr("try_cast(regexp_replace(trim(potencia_kwp), ',', '.') as decimal(12,2))").alias("_potencia_num"),
        F.trim(F.col("energia_injetada_kwh_media_mensal")).alias("energia_raw"),
        F.expr("try_cast(trim(energia_injetada_kwh_media_mensal) as decimal(12,2))").alias("_energia_num"),
        F.trim(F.col("data_conexao")).alias("data_conexao_raw"),
        F.try_to_date(F.trim(F.col("data_conexao"))).alias("data_conexao"),
        F.trim(F.col("modalidade")).alias("modalidade_origem"),
        F.col("source_file"),
        F.col("ingestion_datetime").alias("bronze_ingestion_datetime"),
    )
    .join(df_uc, F.col("id_uc") == F.col("uc_id"), "left")
)

modalidade = mapa_modalidade[F.col("modalidade_origem")]
motivo = (
    F.when(F.col("id_uc").isNull() | (F.col("id_uc") == ""), F.lit("id_uc_nulo"))
    .when(
        F.col("_potencia_num").isNull() | (F.col("_potencia_num") <= 0),
        F.lit("potencia_invalida"),
    )
    .when(
        F.col("_energia_num").isNull() | (F.col("_energia_num") <= 0),
        F.lit("energia_injetada_invalida"),
    )
    .when(F.col("data_conexao").isNull(), F.lit("data_conexao_invalida"))
    .when(modalidade.isNull(), F.lit("modalidade_desconhecida"))
    .when(F.col("uc_id").isNull(), F.lit("uc_inexistente"))
)
df = df.withColumn("modalidade", modalidade).withColumn("motivo_rejeicao", motivo)

df_quarentena = df.filter(F.col("motivo_rejeicao").isNotNull())
df_ok = df.filter(F.col("motivo_rejeicao").isNull())

janela_id = Window.partitionBy("id_uc").orderBy(
    F.col("data_conexao").desc_nulls_last(),
    F.col("bronze_ingestion_datetime").desc_nulls_last(),
)
df_ok = df_ok.withColumn("_rn", F.row_number().over(janela_id)).withColumn(
    "_qtd_id", F.count("*").over(Window.partitionBy("id_uc"))
)
n_descartadas = df_ok.filter(F.col("_rn") > 1).count()
df_uma = df_ok.filter(F.col("_rn") == 1)

potencia_corrigida = (
    F.when(F.col("_potencia_num") > ESCALA_ACIMA_DE, F.col("_potencia_num") / F.lit(1000))
    .otherwise(F.col("_potencia_num"))
    .cast(DecimalType(10, 2))
)
fora_da_janela = (F.col("data_conexao") < F.to_date(F.lit(DATA_MIN))) | (
    F.col("data_conexao") > F.to_date(F.lit(DATA_MAX))
)
anterior_a_ligacao = F.col("data_ligacao").isNotNull() & (
    F.col("data_conexao") < F.col("data_ligacao")
)

df_silver = (
    df_uma.withColumn("potencia_kwp", potencia_corrigida)
    .withColumn("fl_correcao_escala", F.col("_potencia_num") > F.lit(ESCALA_ACIMA_DE))
    .withColumn(
        "energia_injetada_kwh_media_mensal",
        F.col("_energia_num").cast(DecimalType(10, 2)),
    )
    .withColumn(
        "kwh_por_kwp_mes",
        (F.col("_energia_num") / potencia_corrigida).cast(DecimalType(10, 2)),
    )
    .withColumn("fl_data_inconsistente", fora_da_janela | anterior_a_ligacao)
    .withColumn("fl_deduplicado", F.col("_qtd_id") > 1)
    .withColumn("silver_ingestion_datetime", F.current_timestamp())
    .select(
        "id_uc",
        "potencia_kwp_raw",
        "potencia_kwp",
        "fl_correcao_escala",
        "energia_injetada_kwh_media_mensal",
        "kwh_por_kwp_mes",
        "data_conexao",
        "fl_data_inconsistente",
        "modalidade",
        "modalidade_origem",
        "fl_deduplicado",
        "classe",
        "municipio",
        "source_file",
        "bronze_ingestion_datetime",
        "silver_ingestion_datetime",
    )
)

n_bronze = spark.table(bronze_gd).count()
n_silver = df_silver.count()
n_quarentena = df_quarentena.count()

if n_silver + n_quarentena + n_descartadas != n_bronze:
    raise SystemExit(
        f"[ERRO] Bronze {n_bronze} não fecha com Silver {n_silver} + "
        f"quarentena {n_quarentena} + duplicatas {n_descartadas}."
    )

if n_quarentena:
    print(f"[AVISO] quarentena: {n_quarentena} linha(s). A Silver das linhas válidas segue.")
    df_quarentena.groupBy("motivo_rejeicao").count().show(truncate=False)
else:
    print("[OK] quarentena: 0")
if n_descartadas:
    print(
        f"[AVISO] duplicatas descartadas: {n_descartadas}. "
        "fl_deduplicado marca a linha que ficou."
    )


In [0]:
df_quarentena_out = df_quarentena.select(
    "id_uc",
    "potencia_kwp_raw",
    "energia_raw",
    "data_conexao_raw",
    "modalidade_origem",
    "source_file",
    "bronze_ingestion_datetime",
    "motivo_rejeicao",
).withColumn("silver_ingestion_datetime", F.current_timestamp())

(
    df_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_gd)
)
(
    df_quarentena_out.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(quarentena_gd)
)
print(f"Silver gravada: {silver_gd} ({n_silver} linhas)")
print(f"Quarentena gravada: {quarentena_gd} ({n_quarentena} linhas)")


Silver gravada: `v-energy`.silver.tb_geracao_distribuida (3510 linhas)
Quarentena gravada: `v-energy`.silver.tb_geracao_distribuida_quarentena (0 linhas)


In [0]:
gravada = spark.table(silver_gd)
falhas = []

def checar(nome, condicao):
    print(f"[{'OK' if condicao else 'FALHA'}] {nome}")
    if not condicao:
        falhas.append(nome)

checar(
    "uma linha por id_uc",
    gravada.select("id_uc").distinct().count() == gravada.count(),
)
checar(
    "potência entre 0 e 200",
    gravada.filter((F.col("potencia_kwp") <= 0) | (F.col("potencia_kwp") > 200)).count() == 0,
)
checar(
    "injeção positiva",
    gravada.filter(F.col("energia_injetada_kwh_media_mensal") <= 0).count() == 0,
)
checar(
    "modalidade no de-para",
    gravada.filter(~F.col("modalidade").isin(list(MODALIDADE.values()))).count() == 0,
)
potencia_original = F.regexp_replace(F.col("potencia_kwp_raw"), ",", ".").cast("double")
checar(
    "escala coerente com o texto original",
    gravada.filter(
        F.col("fl_correcao_escala") != (potencia_original > ESCALA_ACIMA_DE)
    ).count()
    == 0,
)
marcadas = {
    r["id_uc"]
    for r in gravada.filter(
        F.col("id_uc").isin(UCS_DATA_INCONSISTENTE) & F.col("fl_data_inconsistente")
    )
    .select("id_uc")
    .collect()
}
checar(
    "as cinco UCs com conexão anterior à ligação estão marcadas",
    set(UCS_DATA_INCONSISTENTE) == marcadas,
)

if falhas:
    raise SystemExit("[ERRO] validação da Silver: " + ", ".join(falhas))
print("Validação da tabela gravada fechou.")
gravada.groupBy("modalidade").count().orderBy("modalidade").show(truncate=False)


[OK] uma linha por id_uc
[OK] potência entre 0 e 200
[OK] injeção positiva
[OK] modalidade no de-para
[OK] escala coerente com o texto original
[OK] as cinco UCs com conexão anterior à ligação estão marcadas
Validação da tabela gravada fechou.
+------------------+-----+
|modalidade        |count|
+------------------+-----+
|AUTOCONSUMO_REMOTO|321  |
|COMPARTILHADA     |147  |
|PROPRIA_UC        |3042 |
+------------------+-----+

